# 2. Processing and spatial join

Start the Docker services with `docker compose -f docker/docker-compose.yml up -d --build --wait`, ingest source files, then upload them to HDFS. The commands below run the same Spark jobs as the pipeline script.


In [ ]:
import subprocess

COMPOSE = ['docker', 'compose', '-f', 'docker/docker-compose.yml']
def spark_job(module, *args):
    command = 'cd /workspace && spark-submit --master local[*] /workspace/' + module + ' ' + ' '.join(args)
    return subprocess.run(COMPOSE + ['exec', '-T', 'spark', 'bash', '-lc', command], check=True)

# If needed, run once: python -m src.ingest.upload_to_hdfs inside the spark service.
spark_job('src/process/clean_tiles.py')
spark_job('src/process/geo_join.py', '--boundaries', '/workspace/data/census/boundaries')
spark_job('src/process/feature_engineering.py')


## Inspect the feature table

The join uses tile centroids and TIGER tract polygons. Tile download, upload, and latency averages are weighted by test count. The summed device count is the sum of Ookla's per-tile unique-device counts; the public data does not expose a cross-tile device identifier for tract-level deduplication.


In [ ]:
inspect = 'cd /workspace && hdfs dfs -ls /user/digitaldivide/processed/tract_features'
subprocess.run(COMPOSE + ['exec', '-T', 'spark', 'bash', '-lc', inspect], check=True)
